In [1]:
# --- repo bootstrap: make src/ importable and run from repo root (works wherever the kernel starts) ---
import sys, os
from pathlib import Path
_ROOT = Path.cwd()
while _ROOT != _ROOT.parent and not (_ROOT / 'src').is_dir():
    _ROOT = _ROOT.parent
sys.path.insert(0, str(_ROOT / 'src'))
os.chdir(_ROOT)

In [2]:
# Cell 1 — one Dune API key per account
from dune_fetch import load_api_keys

API_KEYS, KEY_SOURCE = load_api_keys()


  key 1: loaded from DUNE_API_KEY_1
  key 2: loaded from DUNE_API_KEY_2
  key 3: loaded from DUNE_API_KEY_3


In [3]:
# Cell 2 — query registry, split by which Dune account owns each query.
# Names double as CSV filename prefixes (data_validation.TABLE_LABELS, context/api.md).

QUERY_IDS_1 = {
    "reserve_state_rates":          7711042,
    "oracle_price_usd_eth_weth_6h": 8403530,   # 2h-grain; keep "_6h" — TABLE_LABELS keys off it
}
QUERY_IDS_2 = {
    "borrow_repay":      7798273,
    "liquidation":         7798339,
    "flashloan":         7798349,
    "user_account":      7798351,
    "collateral_toggle": 7798372,
}
QUERY_IDS_3 = {
    "reserve_config": 7804264,   # keep this name — normalize.ipynb globs reserve_config_*.csv
    "supply_withdraw":   8595322,
}

QUERY_GROUPS = {1: QUERY_IDS_1, 2: QUERY_IDS_2, 3: QUERY_IDS_3}

# Static per-asset decimals — don't change, already fetched, not re-run here.
DISABLED_QUERY_IDS = {
    "decimal_reference_part1":                   7711171,
    "decimal_reference_part2":                   7711265,
    "decimal_reference_part3":                   7711276,
    "decimal_reference_part4":                   7711290,
    "decimal_reference_part5":                   7711298,
    "decimal_reference_part6":                   7711304,
    "decimal_reference_part9_collateral_toggle": 7711307,
}

_all_ids = [qid for grp in QUERY_GROUPS.values() for qid in grp.values()]
assert len(_all_ids) == len(set(_all_ids)), "duplicate query id across groups"

for _g, _grp in QUERY_GROUPS.items():
    print(f"  group {_g}: {len(_grp)} queries -> {', '.join(_grp)}")
print(f"  {len(DISABLED_QUERY_IDS)} disabled (not attempted by any group)")


  group 1: 2 queries -> reserve_state_rates, oracle_price_usd_eth_weth_6h
  group 2: 5 queries -> borrow_repay, liquidation, flashloan, user_account, collateral_toggle
  group 3: 2 queries -> reserve_config, supply_withdraw
  7 disabled (not attempted by any group)


In [ ]:
# Cell 3 — run settings. Edit this cell between runs.

SETTINGS = dict(
    mode="execute",            # "stored" = read latest, free | "execute" = re-run with window below
    start_date="2025-04-01",   # inclusive
    end_date="2026-03-31",     # exclusive
    dry_run=False,             # True -> print the request only, nothing billed
    max_result_mb=30.0,        # abort the download above this size (compute is already spent)
    performance=None,          # None = let Dune pick; key 1 rejects an explicit tier
    timeout_seconds=480,
    poll_seconds=5,
    param_overrides={
        # As-of snapshot: upper bound only, so start_date is not sent.
        # Every query here is sent the bare 'YYYY-MM-DD' — the SQL supplies the
        # DATE '...' wrapper. reserve_config (7804264) 400s on that today because its
        # end_date parameter is typed *Date* on Dune; set that Type to *Text* in the query
        # editor and this works as written.
        "reserve_config": {"start_date": None},
    },
)

assert SETTINGS["mode"] in {"stored", "execute"}, "mode must be 'stored' or 'execute'"
if SETTINGS["mode"] == "execute":
    assert SETTINGS["start_date"] < SETTINGS["end_date"], "start_date is not before end_date"

print(f"mode={SETTINGS['mode']}")
if SETTINGS["mode"] == "execute":
    print(f"  window {SETTINGS['start_date']} -> {SETTINGS['end_date']} (end exclusive)")
    print(f"  dry_run={SETTINGS['dry_run']}  max={SETTINGS['max_result_mb']} MB")

In [5]:
# Cell 4 — bind registry + keys + settings into run(). Run once.
from dune_fetch import make_runner
from IPython.display import display

run = make_runner(QUERY_GROUPS, API_KEYS, KEY_SOURCE, show=display, **SETTINGS)

print("run(group, tables=None, **overrides) ready — results land in run.results[group]")


run(group, tables=None, **overrides) ready — results land in run.results[group]


In [6]:
# Cell 5 — Account 1 (DUNE_API_KEY_1): reserve_state_rates, oracle_price_usd_eth_weth_6h
run(1)


group 1 (DUNE_API_KEY_1)  mode=execute  2 quer(y/ies)
  window 2025-04-01 -> 2026-03-31 (end exclusive)  dry_run=False

reserve_state_rates (7711042)
        01M1KJD40PD6HGXF56ZZRGHY1Q -> QUERY_STATE_EXECUTING
        01M1KJD40PD6HGXF56ZZRGHY1Q -> QUERY_STATE_COMPLETED
        result: 99143 rows, 20.612 MB, 0.1875 credits to run
  ok      99143 rows -> query_result_data/reserve_state_rates_7711042_2025-04-01_2026-03-31.csv



,time_bucket,asset,liquidity_rate,variable_borrow_rate,stable_borrow_rate,liquidity_index,variable_borrow_index,update_count
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,240460017938305155247568,6837847476521642513159953,0,1001884076755613312382113349,1016138598958428325605394846,5
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,260251357744784236545043,5101483773865203480171851,0,1003301388037686644300001073,1022169551577613794743079565,21
2,2025-04-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,0,45000000000000000000000000,0,1000000000000000000000000000,1123769270328824735501893549,5
3,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,11036850654744743934098848,33221757483825785292984696,0,1038794238037122552428762860,1074808173655674718626217029,5
4,2025-04-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,245661976481994641949144,6911412156870664508856187,0,1000689242880004626658966787,1007202064164604922005757426,2
...,...,...,...,...,...,...,...,...
99138,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,16593501039374015256199383,22330587822744240960942379,0,1061964462866918475372408504,1094463563874997554894048802,72
99139,2026-03-30 22:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,7920771772666183873927915,22249679460930311021372761,0,1013738814497428977415038756,1031589295195676559422282822,2
99140,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,16826941441274199728831,3051442240019971988095075,0,1000194592987104314559358298,1004010540209208280003807314,6
99141,2026-03-30 22:00:00.000 UTC,0xcd5fe23c85820f7b72d0926fc9b05b43e359b7ee,1068123552227902879983,10045110833786270265827753,0,1000998384578938030332755056,1027719781869594642753949454,1


oracle_price_usd_eth_weth_6h (8403530)
        01M1KJDSK51TRMB90TPA4B55VF -> QUERY_STATE_PENDING
        01M1KJDSK51TRMB90TPA4B55VF -> QUERY_STATE_EXECUTING
        01M1KJDSK51TRMB90TPA4B55VF -> QUERY_STATE_COMPLETED
        result: 206330 rows, 16.17 MB, 21.6025 credits to run
  ok      206330 rows -> query_result_data/oracle_price_usd_eth_weth_6h_8403530_2025-04-01_2026-03-31.csv



,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,price_points
0,2025-04-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.18799116666666665,0.0001028974115429241,2
1,2025-04-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,82427.63208333333,45.116953651426286,2
2,2025-04-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.0825615833333333,0.0005925425494564018,2
3,2025-04-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.995833333333334,0.003281834057004702,2
4,2025-04-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,82554.12833333333,45.18619131400641,2
...,...,...,...,...,...,...,...
206325,2026-03-30 22:00:00,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,18,1.0001571666666667,0.0004929371852630663,2
206326,2026-03-30 22:00:00,0xdefa4e8a7bcba345f687a2f1456f5edd9ce97202,KNC,18,0.14840454166666667,0.00007314225413201651,2
206327,2026-03-30 22:00:00,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6,1.0001280416666667,0.0004929229108403565,2
206328,2026-03-30 22:00:00,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,18,2155.2833333333333,1.0622436916572782,2



group 1: 2/2 tables (execute mode)
  execution credits spent on account 1: 21.79
  manifest: query_result_data/_runs/run_20260903T120505Z.json


{'tables': {'reserve_state_rates':                        time_bucket  \
  0      2025-04-01 00:00:00.000 UTC   
  1      2025-04-01 00:00:00.000 UTC   
  2      2025-04-01 00:00:00.000 UTC   
  3      2025-04-01 00:00:00.000 UTC   
  4      2025-04-01 00:00:00.000 UTC   
  ...                            ...   
  99138  2026-03-30 22:00:00.000 UTC   
  99139  2026-03-30 22:00:00.000 UTC   
  99140  2026-03-30 22:00:00.000 UTC   
  99141  2026-03-30 22:00:00.000 UTC   
  99142  2026-03-30 22:00:00.000 UTC   
  
                                              asset              liquidity_rate  \
  0      0x1f9840a85d5af5bf1d1762f925bdaddc4201f984    240460017938305155247568   
  1      0x2260fac5e5542a773aa44fbcfedf7c193bc2c599    260251357744784236545043   
  2      0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f                           0   
  3      0x4c9edd5852cd905f086c759e8383e09bff1e68b3  11036850654744743934098848   
  4      0x514910771af9ca656af840dff83e8264ecf986ca    24566197648199

In [11]:
# Cell 6 — Account 2 (DUNE_API_KEY_2): supply_withdraw, borrow_repay, liquidation,
#           flashloan, user_account, collateral_toggle
run(2, ["user_account", "collateral_toggle"])


group 2 (DUNE_API_KEY_2)  mode=execute  2 quer(y/ies)
  window 2025-04-01 -> 2026-03-31 (end exclusive)  dry_run=False

user_account (7798351)
        01M1KJXGA6EGK87CAN9SXBBFT8 -> QUERY_STATE_EXECUTING
        01M1KJXGA6EGK87CAN9SXBBFT8 -> QUERY_STATE_COMPLETED
        result: 2174 rows, 0.166 MB, 0.240394737 credits to run
        exact big integers, no action needed: avg_available_borrows_base, avg_total_collateral_base, avg_total_debt_base
        PRECISION LOSS — arrived as DOUBLE, low digits already discarded by Dune:
          max_health_factor: 497 value(s), up to 78 digits (largest 1.157920892373162E+77)
          min_health_factor: 180 value(s), up to 78 digits (largest 1.157920892373162E+77)
          fix in the SQL for user_account, not here: emit the raw integer as CAST(max_health_factor AS VARCHAR) and do any AVG/SUM/division in Python.
  ok      2174 rows -> query_result_data/user_account_7798351_2025-04-01_2026-03-31.csv



,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,min_health_factor,max_health_factor,sampled_user_count,account_data_call_count
0,2025-04-01 02:00:00.000 UTC,14915209250228770,4997478545129555,6710960716300029,8100,7850,2417483013400732000,2417483013400732000,1,1
1,2025-04-01 04:00:00.000 UTC,15063247131465796,5192650167817343,6631998830383308,8100,7850,2349711569654212000,2349711569654212000,1,1
2,2025-04-01 08:00:00.000 UTC,126248699850,47721495282,46965029606,7800,7500,2063514257067784200,2063514257067784200,1,1
3,2025-04-01 12:00:00.000 UTC,10477125182851400,6422148304017836,3321578116033967,9500,9300,1549834798657927200,1549834798657927200,1,1
4,2025-04-02 12:00:00.000 UTC,10477712648964236,6422465330550661,3321807432986078,9500,9300,1549845192494419500,1549845192494419500,1,1
...,...,...,...,...,...,...,...,...,...,...
2169,2026-03-29 22:00:00.000 UTC,8784238053418.7,6585791731526,484928549681.6,7420,7190,1076588948005915400,1.157920892373162E+77,6,10
2170,2026-03-30 04:00:00.000 UTC,3156979626653,2259402309921,281966289534.5,8300,8050,1133687179121993100,1190561214058354200,1,2
2171,2026-03-30 10:00:00.000 UTC,153939987148075,112974558959404,25571429473863,9300,9000,1239787133436532200,1299990045972200200,1,2
2172,2026-03-30 16:00:00.000 UTC,758136464,386415170,318651741,9500,9300,1863875170325223000,1863875170325223000,1,1


collateral_toggle (7798372)
        01M1KJXSAFW124F9R6SG97HBSX -> QUERY_STATE_EXECUTING
        01M1KJXSAFW124F9R6SG97HBSX -> QUERY_STATE_COMPLETED
        result: 60436 rows, 5.245 MB, 0.064617648 credits to run
  ok      60436 rows -> query_result_data/collateral_toggle_7798372_2025-04-01_2026-03-31.csv



,time_bucket,asset,asset_symbol,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users,latest_collateral_toggle_block
0,2025-04-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,None,1,1,1,1,22170819
1,2025-04-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,None,9,4,8,3,22170740
2,2025-04-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,None,1,0,1,0,22170829
3,2025-04-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,None,0,1,0,1,22170539
4,2025-04-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,None,2,2,2,2,22170805
...,...,...,...,...,...,...,...,...
60431,2026-03-30 22:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,None,1,1,1,1,24773360
60432,2026-03-30 22:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,None,10,10,9,10,24773855
60433,2026-03-30 22:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,None,13,8,11,6,24773814
60434,2026-03-30 22:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,None,1,0,1,0,24773477



group 2: 2/2 tables (execute mode)
  execution credits spent on account 2: 0.305012385
  manifest: query_result_data/_runs/run_20260903T121216Z.json


{'tables': {'user_account':                       time_bucket avg_total_collateral_base  \
  0     2025-04-01 02:00:00.000 UTC         14915209250228770   
  1     2025-04-01 04:00:00.000 UTC         15063247131465796   
  2     2025-04-01 08:00:00.000 UTC              126248699850   
  3     2025-04-01 12:00:00.000 UTC         10477125182851400   
  4     2025-04-02 12:00:00.000 UTC         10477712648964236   
  ...                           ...                       ...   
  2169  2026-03-29 22:00:00.000 UTC           8784238053418.7   
  2170  2026-03-30 04:00:00.000 UTC             3156979626653   
  2171  2026-03-30 10:00:00.000 UTC           153939987148075   
  2172  2026-03-30 16:00:00.000 UTC                 758136464   
  2173  2026-03-30 18:00:00.000 UTC               94576778288   
  
       avg_total_debt_base avg_available_borrows_base  \
  0       4997478545129555           6710960716300029   
  1       5192650167817343           6631998830383308   
  2            47721

In [15]:
# Cell 7 — Account 3 (DUNE_API_KEY_3): reserve_config
run(3, ["reserve_config"])


group 3 (DUNE_API_KEY_3)  mode=execute  1 quer(y/ies)
  window 2025-04-01 -> 2026-03-31 (end exclusive)  dry_run=False

reserve_config (7804264)
  FAILED  HTTPError: 400 Bad Request on POST /query/7804264/execute: invalid query parameters: The parameter 'end_date' doesn't have a valid date: '2026-03-31'


group 3: 0/1 tables (execute mode)
  failures:
    reserve_config: 7804264: HTTPError: 400 Bad Request on POST /query/7804264/execute: invalid query parameters: The parameter 'end_date' doesn't have a valid date: '2026-03-31'


{'tables': {},
 'metas': [],
 'failures': {'reserve_config': "7804264: HTTPError: 400 Bad Request on POST /query/7804264/execute: invalid query parameters: The parameter 'end_date' doesn't have a valid date: '2026-03-31'"}}